# Proyecto LSP — Fase tiempo real (1/2): demo continuo sobre VIDEO

Pipeline completo del video objetivo, procesando **videos reales** (no caché):

**Video de cada seña → MediaPipe Holistic → reconocedor (top-5, 'locking in') → LLM arma la frase → voz.**

Usa los **videos de referencia** del dataset (`VIDEOS_REF.zip`, solo 24 MB, 1 video por clase)
y el modelo entrenado en el notebook 03. Aquí SÍ se corre MediaPipe en vivo sobre cada video,
así que valida toda la cadena de extracción + inferencia + LLM + TTS.

> **Nota honesta:** los videos de referencia (`rec*`) fueron vistos en el entrenamiento,
> así que el acierto aquí es **optimista**. El número real de generalización es el del test:
> top-5 ≈ 81%. Este demo sirve para mostrar el FLUJO, no para medir precisión.

La webcam en vivo (parte 2/2) corre en tu PC con Python 3.11/3.12 (Holistic no existe en 3.13).

**Pre-requisitos:** haber corrido el notebook 03 (modelo + clases en Drive) y el secreto
de Colab `GEMINI_API_KEY`.

## Setup — repo, dependencias y Drive

Instala MediaPipe (para extraer landmarks del video) + TF + Gemini + gTTS. Desinstala
JAX por el conflicto ml_dtypes con TF 2.18.

In [ ]:
%cd /content
!rm -rf /content/Proyecto_Percepcion
!git clone -q https://github.com/Jtarazona00/Proyecto_Percepcion.git /content/Proyecto_Percepcion
%cd /content/Proyecto_Percepcion

import os, sys
os.environ['DATASET'] = 'pucp305'
sys.path.insert(0, '/content/Proyecto_Percepcion')

!pip install -q tensorflow==2.18.0 mediapipe==0.10.14 google-generativeai gTTS
!pip uninstall -q -y jax jaxlib

from google.colab import drive
drive.mount('/content/drive')

## Paso 1 — Descargar los videos de referencia (24 MB) de Zenodo

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('/content/pucp305'); DATA.mkdir(exist_ok=True)
VIDEOS_DIR = Path('/content/VIDEOS_REF')
ZEN = 'https://zenodo.org/records/13691887/files'

REF_CSV  = DATA / 'videos_ref_annotations.csv'
VID_ZIP  = DATA / 'VIDEOS_REF.zip'
!wget -c -q -O {str(REF_CSV)} '{ZEN}/videos_ref_annotations.csv?download=1'
!wget -c -q -O {str(VID_ZIP)} '{ZEN}/VIDEOS_REF.zip?download=1'
if not VIDEOS_DIR.exists():
    !unzip -n -q {str(VID_ZIP)} -d /content

# Mapa LABEL -> ruta del video de referencia (rec*.mp4)
ref = pd.read_csv(REF_CSV)  # FILENAME,CLASS_ID,LABEL
label2video = {row.LABEL: VIDEOS_DIR / f'{row.FILENAME}.mp4' for row in ref.itertuples()}
print(f'Videos de referencia: {len(list(VIDEOS_DIR.glob("*.mp4")))}')
print('Ejemplo:', list(label2video.items())[110])

## Paso 2 — Cargar modelo y clases

In [ ]:
import json
import tensorflow as tf
import config

MODELS_DRIVE = Path('/content/drive/MyDrive/PUCP305_models')
model = tf.keras.models.load_model(MODELS_DRIVE / 'modelo_pucp305_final.keras', compile=False)
with open(MODELS_DRIVE / 'classes_pucp305.json', encoding='utf-8') as f:
    clases = json.load(f)
config.set_classes(clases)
print(f'Modelo cargado. {len(clases)} clases. FRAMES={config.FRAMES}.')

## Paso 3 — API key de Gemini

In [ ]:
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    import getpass
    GEMINI_API_KEY = getpass.getpass('GEMINI_API_KEY: ')
print('Key lista.' if GEMINI_API_KEY else 'FALTA la key.')

## Paso 4 — Señar la frase (procesar los videos en orden)

Define la secuencia de señas a 'decir'. Para cada una se procesa su video de referencia
con MediaPipe y se obtiene el top-5 (simulando el 'locking in' con la confianza).
Cambia `FRASE` por cualquier combinación de las 299 glosas (deben existir como LABEL).

In [ ]:
from src.inferencia.inferencia_continua import construir_secuencia_glosas

FRASE = ['DOLOR-DE-CABEZA', 'FIEBRE2', 'MAREO']

items = []
for label in FRASE:
    ruta = label2video.get(label)
    assert ruta and ruta.exists(), f'No hay video de referencia para {label!r}'
    items.append((label, ruta))

secuencia, reporte = construir_secuencia_glosas(items, model, clases, top_k=5)
print('\nSecuencia de candidatos (top-5) que recibe el LLM:')
for slot in secuencia:
    print('  ', slot)

## Paso 5 — El LLM arma la frase

In [ ]:
from src.inferencia.llm_frase import armar_frase

frase = armar_frase(secuencia, api_key=GEMINI_API_KEY, contexto='hospitalario')
print('Señas objetivo:', FRASE)
print('Frase generada:', frase)

## Paso 6 — La frase en voz

In [ ]:
from src.inferencia.tts import sintetizar_archivo
from IPython.display import Audio

ruta = sintetizar_archivo(frase, 'frase.mp3', lang='es', tld='com.mx')
Audio(ruta, autoplay=True)